### Exploratory Notebook

In [32]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, LongType, IntegerType
from pyspark.sql.functions import from_json, col
from pyspark.sql.functions import regexp_replace


################################################
# Schemas 

# Define the schema for the JSON data
event_schema = StructType([
    StructField("age_of_insured", IntegerType(), True),
    StructField("coverage_amount", DoubleType(), True),
    StructField("customer_id", StringType(), True),
    StructField("event_timestamp", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("policy_id", StringType(), True),
    StructField("policy_type", StringType(), True),
    StructField("premium_amount", DoubleType(), True),
    StructField("region", StringType(), True),
])

# Define the schema for the policy_type field
policy_type_schema = StructType([
                StructField("type", StringType(), True),
                StructField("brand", StringType(), True),
])

########################################################
# Load and clean data
     
# Read the JSON data from the specified path and apply the schema
invalid_json_df =   spark \
                    .read\
                    .format("json")\
                    .schema(event_schema) \
                    .load("/Volumes/ageas/bronze/files")

# Fix the invalid JSON in the policy_type field by replacing single quotes with double quotes
valid_json_df = invalid_json_df.withColumn(
    "policy_type",
    regexp_replace(col("policy_type"), "'", "\"")
)

# Replace policy_type string with a valid JSON object
valid_json_df = valid_json_df.withColumn(
    "policy_type_object",
    from_json(col("policy_type"), policy_type_schema)
)


valid_json_df = valid_json_df.selectExpr(
    "age_of_insured",
    "coverage_amount",
    "customer_id",
    "event_timestamp",
    "event_type",
    "policy_id",
    "policy_type_object.type AS policy_type",
    "policy_type_object.brand AS policy_brand",
    "premium_amount",
    "region"
)

display(valid_json_df)

,age_of_insured,coverage_amount,customer_id,event_timestamp,event_type,policy_id,policy_type,policy_brand,premium_amount,region
0,56,6.102073e+04,CUS-70018,2024-01-21T10:40:41.087Z,claim,POL-58051,auto,LifeSecure,938.88,North
1,62,3.132927e+04,CUS-56010,2024-07-24T01:56:56.088Z,purchase,POL-84577,life,InsureCorp,687.09,East
2,39,2.681022e+04,CUS-4315,2024-12-22T10:25:58.088Z,purchase,POL-58297,life,InsureCorp,1108.62,West
3,66,8.305563e+04,CUS-37602,2024-04-23T23:51:29.088Z,claim,POL-21521,auto,LifeSecure,1766.21,South
4,71,3.844622e+04,CUS-84286,2024-09-30T03:07:48.088Z,purchase,POL-69237,auto,InsureCorp,905.40,North
5,55,2.991422e+04,CUS-39540,2024-03-25T20:25:51.088Z,purchase,POL-60927,None,None,581.55,North
6,58,4.399557e+04,CUS-89696,2024-08-13T06:52:57.088Z,purchase,POL-47666,health,LifeSecure,1272.79,South
7,60,4.442296e+04,CUS-8242,2024-03-16T11:19:41.088Z,claim,POL-16907,None,None,793.39,North
8,75,1.404010e+04,CUS-52665,2024-01-08T18:23:04.088Z,cancellation,POL-54380,home,InsureCorp,1714.52,West
9,71,4.332432e+04,CUS-94911,2024-03-27T09:38:29.088Z,cancellation,POL-45657,health,ProtectPlus,525.92,West
